# Ingesting network traffic into the lakehouse

The **Network Traffic Monitor** add-on captures full packets on the Home
Assistant host and uploads two things per rotation, straight to MinIO — no
Airflow DAG pulls this in the way the trackers' data arrives, it is already
sitting in the `raw` bucket by the time you read this:

- `network_traffic/<date>/<label>-<timestamp>.jsonl` — one JSON record per
  packet: timestamps, the 5-tuple, and DNS query / TLS SNI / plaintext HTTP
  host+path where the protocol exposed them. **This is what this notebook
  ingests.**
- `network_traffic/<date>/<label>-<timestamp>.pcap` — the full capture behind
  it, for the rare deep-dive `network-traffic/app/extract.py`'s field
  extraction did not cover. Not read here; open it with Wireshark if you ever
  need it.

This is deliberately a **plain append**, not the trackers' MERGE-on-`seq`
pattern in `lakehouse.py` — a packet is an immutable fact the moment it is
captured, never edited or deleted, so there is nothing to reconcile. That
also means **re-running the write cell below over files it has already
ingested duplicates rows** — see *Where to go next* for what changes once
this is worth turning into a scheduled job.

In [ ]:
import os, sys
sys.path.insert(0, "/share/pipeline-airflow/lib")

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.remote(
    os.environ.get("SPARK_CONNECT_URL", "sc://172.30.32.1:15002")
).getOrCreate()

LAKEHOUSE_ROOT = os.environ.get("LAKEHOUSE_ROOT", "s3a://lakehouse")
RAW_PATH = "s3a://raw/network_traffic/*/*.jsonl"
TABLE_PATH = f"{LAKEHOUSE_ROOT}/network_traffic/packets"

## The raw shape

Matches `record_from_fields()` in `network-traffic/app/extract.py` exactly —
one row per line, already flat, nothing wrapped in a JSON-string payload the
way the trackers' change-feed rows are. An explicit schema rather than
inference for the same reason `lakehouse.py` uses one: two batches disagreeing
about a type would otherwise fail the write, and a reader can ignore a column
it doesn't care about but not one that was silently guessed wrong.

In [ ]:
PACKET_SCHEMA = (
    "time double, src_ip string, dst_ip string, ip_proto string, "
    "length long, src_port long, dst_port long, "
    "dns_query string, tls_sni string, http_host string, http_uri string, "
    "protocol string, info string"
)

raw = (
    spark.read.schema(PACKET_SCHEMA).json(RAW_PATH)
    # The date the add-on already put in the object key (see object_keys() in
    # network-traffic/app/uploader.py) rather than re-deriving one from `time`
    # — robust to a null or slightly-off timestamp on a malformed row, and
    # it's the same value that decided which file the row came from.
    .withColumn("date", F.regexp_extract(F.input_file_name(),
                                          r"network_traffic/(\d{4}-\d{2}-\d{2})/", 1))
)

print("rows:", raw.count())
raw.show(10, truncate=60)

## A quick look before ingesting

Worth doing once before trusting a write: confirms the capture is actually
seeing something, and which protocols/hosts dominate before you have a whole
Delta table to query instead.

In [ ]:
raw.groupBy("protocol").count().orderBy(F.col("count").desc()).show(15)

In [ ]:
# DNS queries and TLS SNI are the two fields that answer "what was this
# device talking to" for traffic this capture can't otherwise read the
# content of — see DOCS.md's note on what's visible for encrypted traffic.
(raw.where(F.col("dns_query").isNotNull())
    .groupBy("dns_query").count().orderBy(F.col("count").desc()).show(15, truncate=False))

(raw.where(F.col("tls_sni").isNotNull())
    .groupBy("tls_sni").count().orderBy(F.col("count").desc()).show(15, truncate=False))

In [ ]:
# Byte volume by source — length is per-packet, so this is total bytes sent,
# not a rate; divide by the time span covered if you want MB/s.
(raw.groupBy("src_ip")
    .agg(F.sum("length").alias("bytes"), F.count("*").alias("packets"))
    .orderBy(F.col("bytes").desc())
    .withColumn("mb", F.round(F.col("bytes") / 1048576, 2))
    .show(15))

## Writing it into the lakehouse

Partitioned by `date`, matching how the raw bucket itself is already laid
out — a query scoped to a day range prunes to exactly those files instead
of scanning everything. Append mode: see the caveat at the top about
re-running this over the same files.

In [ ]:
(raw.write.format("delta")
    .mode("append")
    .partitionBy("date")
    .save(TABLE_PATH))

print(f"wrote to {TABLE_PATH}")

In [ ]:
# Read it back the same way lakehouse.py's raw()/table() would, to confirm
# it actually landed.
written = spark.read.format("delta").load(TABLE_PATH)
print("rows in the table:", written.count())
written.groupBy("date").count().orderBy("date").show()

---

### Where to go next

This notebook is the manual, "does this work at all" step. Once you trust
the shape of the data, the natural next move is the same one every tracker
already took in `pipeline-airflow/dags/trackers_ingest.py`: a scheduled DAG
that

1. tracks which raw files have already been ingested (a watermark, the way
   `pipeline_meta.source_watermark` does for the trackers — keyed on object
   key here, since there's no `seq` in this data to watermark on), and
2. appends only the new ones, so a scheduled run is idempotent instead of
   duplicating rows the way re-running this notebook's write cell does.

`network-traffic/config.yaml`'s `datalake_retention_days` (default 7) expires
the *raw* JSONL/pcap pairs in MinIO on its own schedule — whatever you want
to keep longer than that needs to already be in this Delta table before the
lifecycle rule catches up to it.